# Import Raw nflverse Parquet Files To Bronze Tables

Run this notebook in Microsoft Fabric with your target Lakehouse attached.

Prerequisites:

- Upload local `nflverse_local/raw/nflverse/` to Lakehouse `Files/raw/nflverse/`.
- Upload local `nflverse_local/manifest/` to Lakehouse `Files/manifest/`.
- Use a schema-enabled Lakehouse. This notebook creates the `bronze` schema if it does not already exist.

The notebook writes managed Delta tables under the `bronze` schema while preserving source columns and adding ingestion metadata.

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp, input_file_name, lit

RAW_ROOT = "Files/raw/nflverse"
MANIFEST_ROOT = "Files/manifest"
BRONZE_SCHEMA = "bronze"
WRITE_MODE = "overwrite"

ACQUISITION_BATCH_ID = "nflverse_1999_2025"
NOTEBOOK_RUN_UTC = datetime.now(timezone.utc).isoformat()

print(f"Notebook run UTC: {NOTEBOOK_RUN_UTC}")
print(f"Raw root: {RAW_ROOT}")
print(f"Bronze schema: {BRONZE_SCHEMA}")

## Create Bronze Schema

The Lakehouse is assumed to be attached to this notebook. If this cell fails on `CREATE SCHEMA`, create or switch to a schema-enabled Lakehouse.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{BRONZE_SCHEMA}`")
spark.sql(f"SHOW TABLES IN `{BRONZE_SCHEMA}`").show(200, truncate=False)

## Source-To-Table Map

Each path is relative to `Files/raw/nflverse`. Weekly rosters begin in 2002 because that is the first season available from `nflreadpy`; the other datasets cover 1999 through 2025.

In [ ]:
TABLE_SPECS = [
    {
        "table": "nflverse_pbp",
        "source": "pbp",
        "description": "Raw nflverse play-by-play data, 1999-2025.",
    },
    {
        "table": "nflverse_schedules",
        "source": "schedules",
        "description": "Raw nflverse game schedule and result data, 1999-2025.",
    },
    {
        "table": "nflverse_teams",
        "source": "teams",
        "description": "Raw nflverse team metadata.",
    },
    {
        "table": "nflverse_players",
        "source": "players",
        "description": "Raw nflverse player metadata.",
    },
    {
        "table": "nflverse_rosters",
        "source": "rosters",
        "description": "Raw nflverse season roster data, 1999-2025.",
    },
    {
        "table": "nflverse_rosters_weekly",
        "source": "rosters_weekly",
        "description": "Raw nflverse weekly roster data, 2002-2025.",
    },
    {
        "table": "nflverse_player_stats_weekly",
        "source": "player_stats_weekly",
        "description": "Raw nflverse weekly player stats, 1999-2025.",
    },
    {
        "table": "nflverse_team_stats_weekly",
        "source": "team_stats_weekly",
        "description": "Raw nflverse weekly team stats, 1999-2025.",
    },
    {
        "table": "nflverse_player_stats_reg",
        "source": "player_stats_season/summary_level=reg",
        "description": "Raw nflverse regular-season player stats, 1999-2025.",
    },
    {
        "table": "nflverse_player_stats_post",
        "source": "player_stats_season/summary_level=post",
        "description": "Raw nflverse postseason player stats, 1999-2025.",
    },
    {
        "table": "nflverse_team_stats_reg",
        "source": "team_stats_season/summary_level=reg",
        "description": "Raw nflverse regular-season team stats, 1999-2025.",
    },
    {
        "table": "nflverse_team_stats_post",
        "source": "team_stats_season/summary_level=post",
        "description": "Raw nflverse postseason team stats, 1999-2025.",
    },
]

for spec in TABLE_SPECS:
    print(f"{BRONZE_SCHEMA}.{spec['table']} <- {RAW_ROOT}/{spec['source']}")

## Import Helpers

`recursiveFileLookup=true` intentionally reads Parquet files beneath Hive-style folders such as `season=1999` without adding a second inferred partition column. The raw files already carry source columns such as `season` where available.

In [ ]:
def full_table_name(table_name: str) -> str:
    return f"{BRONZE_SCHEMA}.{table_name}"


def quoted_table_name(table_name: str) -> str:
    return f"`{BRONZE_SCHEMA}`.`{table_name}`"


def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def source_path(spec: dict) -> str:
    return f"{RAW_ROOT}/{spec['source']}"


def read_raw_parquet(path: str) -> DataFrame:
    return (
        spark.read
        .option("recursiveFileLookup", "true")
        .option("mergeSchema", "true")
        .parquet(path)
    )


def add_bronze_metadata(df: DataFrame, spec: dict) -> DataFrame:
    return (
        df.withColumn("_bronze_ingested_at_utc", current_timestamp())
        .withColumn("_source_file_path", input_file_name())
        .withColumn("_source_system", lit("nflreadpy"))
        .withColumn("_source_dataset", lit(spec["table"]))
        .withColumn("_acquisition_batch_id", lit(ACQUISITION_BATCH_ID))
    )


def write_bronze_table(spec: dict) -> dict:
    path = source_path(spec)
    table_name = full_table_name(spec["table"])
    quoted_name = quoted_table_name(spec["table"])

    print(f"Reading {path}")
    df = add_bronze_metadata(read_raw_parquet(path), spec).persist()
    source_rows = df.count()
    source_columns = len(df.columns)

    print(f"Writing {table_name}: {source_rows:,} rows, {source_columns:,} columns")
    (
        df.write
        .format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    df.unpersist()

    table_rows = spark.table(table_name).count()
    spark.sql(f"COMMENT ON TABLE {quoted_name} IS {sql_string(spec['description'])}")

    return {
        "schema": BRONZE_SCHEMA,
        "table": spec["table"],
        "source_path": path,
        "source_rows": source_rows,
        "table_rows": table_rows,
        "columns": source_columns,
        "row_count_matches": source_rows == table_rows,
    }

## Write Bronze Tables

This cell overwrites the Bronze tables. Re-running it is safe when you want the Bronze layer to exactly match the uploaded raw files.

In [ ]:
results = []

for spec in TABLE_SPECS:
    results.append(write_bronze_table(spec))

summary_df = spark.createDataFrame(results)
display(summary_df.orderBy("table"))

## Verify Bronze Load

This cell fails the notebook if any table row count differs from the source-file row count.

In [ ]:
failed = [row for row in results if not row["row_count_matches"]]

if failed:
    raise ValueError(f"Bronze import row-count mismatch: {failed}")

print("All Bronze row counts match source Parquet reads.")
spark.sql(f"SHOW TABLES IN `{BRONZE_SCHEMA}`").show(200, truncate=False)

## Optional: Inspect Manifests

Run this after uploading `nflverse_local/manifest/` to `Files/manifest/`.

In [ ]:
manifest_df = spark.read.option("multiLine", "true").json(f"{MANIFEST_ROOT}/acquisition_manifest.json")
quality_df = spark.read.option("multiLine", "true").json(f"{MANIFEST_ROOT}/quality_report.json")

display(manifest_df.select("fetched_at_utc", "seasons", "team_focus", "availability_notes"))
display(quality_df.select("validated_at_utc", "passed", "seasons_expected", "team_focus"))